# Crane PID AI — Data Exploration & Model Training

Notebook for exploring test_results.csv and training the PID predictor model.
Run cells in order.


In [ ]:
# Section 1 — Load and explore data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/test_results.csv')
print(f"Total rows: {len(df)}")
print(f"Status counts:\n{df['status'].value_counts()}\n")
print(df.describe())


In [ ]:
# Section 2 — Score and metric distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['score'].hist(ax=axes[0], bins=50, color='teal', edgecolor='black')
axes[0].set_title('Score distribution (lower = better)')

df['t_settle'].hist(ax=axes[1], bins=50, color='steelblue', edgecolor='black')
axes[1].set_title('Settle time [s]')

df['overshoot_deg'].hist(ax=axes[2], bins=50, color='coral', edgecolor='black')
axes[2].set_title('Overshoot [°]')

plt.tight_layout()
plt.show()


In [ ]:
# Section 3 — Heatmap: Kp vs Kd for a specific scenario
top = df[df['status'] == 'ok'].nsmallest(50, 'score')
print(top[['L', 'm', 'Kp', 'Ki', 'Kd', 't_settle', 'overshoot_deg', 'score']].head(10))

pivot = df[(df['L'] == 10) & (df['m'] == 50)].pivot_table(
    values='score', index='Kd', columns='Kp', aggfunc='min')

if not pivot.empty:
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, cmap='RdYlGn_r', annot=True, fmt='.2f', linewidths=0.5)
    plt.title('Score: Kp vs Kd (L=10m, m=50kg) — green=better')
    plt.show()
else:
    print('No data for L=10, m=50 — adjust filters')


In [ ]:
# Section 4 — Train the model
import sys
sys.path.insert(0, '../ai-service')
from model import PIDPredictor

p = PIDPredictor()
stats = p.train('../data/test_results.csv')

print(f"\nTraining complete — {stats['n_total']} rows used")
print(f"Trained at: {stats['trained_at']}\n")
for param, m in stats['metrics'].items():
    stars = '★' * int(m['r2'] * 5)
    print(f"  {param}: R²={m['r2']:.3f} {stars}  MAE={m['mae']:.4f}  "
          f"(train={m['n_train']}, test={m['n_test']})")


In [ ]:
# Section 5 — Validate predictions on sample scenarios
cases = [
    {'L': 5,  'm': 30,  'wind_speed': 5,  'wind_dir_deg': 45},
    {'L': 10, 'm': 50,  'wind_speed': 8,  'wind_dir_deg': 135},
    {'L': 15, 'm': 100, 'wind_speed': 12, 'wind_dir_deg': 270},
    {'L': 20, 'm': 200, 'wind_speed': 15, 'wind_dir_deg': 0},
]

print(f"{'L':>4} {'m':>5} {'Wind':>5} | {'Kp':>7} {'Ki':>7} {'Kd':>7} | {'Conf':>6} {'Model'}")
print('-' * 60)
for c in cases:
    r = p.predict(**c)
    print(f"{c['L']:>4} {c['m']:>5} {c['wind_speed']:>5} | "
          f"{r['Kp']:>7.2f} {r['Ki']:>7.3f} {r['Kd']:>7.2f} | "
          f"{r['confidence']:>6.3f} {r['model']}")


In [ ]:
# Section 6 — Feature importances
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, target in zip(axes, ['Kp', 'Ki', 'Kd']):
    imp = p.models[target]['model'].feature_importances_
    colors = ['steelblue'] * len(PIDPredictor.FEATURE_COLS)
    ax.barh(PIDPredictor.FEATURE_COLS, imp, color=colors, edgecolor='black')
    ax.set_title(f'Feature importance — {target}')
    ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
